# 🧹 Preprocessing danych i Wektoryzacja TF-IDF

Ten notatnik zawiera proces wczytywania, wstępnej analizy oraz przygotowania danych tekstowych ze zbioru **20 Newsgroups** do modelowania tematycznego za pomocą algorytmu KMeans.

## 🔍 Eksploracja pierwotnych (surowych) danych

Zanim przejdziemy do oczyszczania tekstu, zobaczmy jak wyglądają pierwotne dokumenty w bazie danych 20 Newsgroups (razem z nagłówkami e-mail, stopkami i cytatami).

In [3]:
import os
from sklearn.datasets import fetch_20newsgroups

# Pobranie surowych danych (z nagłówkami, stopkami i cytatami)
print("Pobieranie surowych danych...")
raw_dataset = fetch_20newsgroups(subset='all')
print(f"Liczba dokumentów w pełnym zbiorze: {len(raw_dataset.data)}")
print(f"Kategorie tematyczne ({len(raw_dataset.target_names)}):")
for name in raw_dataset.target_names:
    print(f" - {name}")

print("\n" + "="*50)
print("PRZYKŁADOWY SUROWY DOKUMENT (Z NAGŁÓWKAMI):")
print("="*50)
print(raw_dataset.data[0][:1200]) # Pokazujemy pierwsze 1200 znaków surowego e-maila


Pobieranie surowych danych...
Liczba dokumentów w pełnym zbiorze: 18846
Kategorie tematyczne (20):
 - alt.atheism
 - comp.graphics
 - comp.os.ms-windows.misc
 - comp.sys.ibm.pc.hardware
 - comp.sys.mac.hardware
 - comp.windows.x
 - misc.forsale
 - rec.autos
 - rec.motorcycles
 - rec.sport.baseball
 - rec.sport.hockey
 - sci.crypt
 - sci.electronics
 - sci.med
 - sci.space
 - soc.religion.christian
 - talk.politics.guns
 - talk.politics.mideast
 - talk.politics.misc
 - talk.religion.misc

PRZYKŁADOWY SUROWY DOKUMENT (Z NAGŁÓWKAMI):
From: Mamatha Devineni Ratnam <mr47+@andrew.cmu.edu>
Subject: Pens fans reactions
Organization: Post Office, Carnegie Mellon, Pittsburgh, PA
Lines: 12
NNTP-Posting-Host: po4.andrew.cmu.edu



I am sure some bashers of Pens fans are pretty confused about the lack
of any kind of posts about the recent Pens massacre of the Devils. Actually,
I am  bit puzzled too and a bit relieved. However, I am going to put an end
to non-PIttsburghers' relief with a bit of prai

## ⚙️ Preprocessing i Wektoryzacja TF-IDF

W celu przeprowadzenia rzetelnej klasteryzacji tematycznej, usuwamy metadane e-maili (nagłówki, stopki, cytaty). Następnie oczyszczamy tekst za pomocą lematyzacji i usuwania stop-words (moduł ), po czym tworzymy macierz TF-IDF.

In [6]:
import sys
import os
import nltk

# Dodanie lokalnej ścieżki NLTK
nltk.data.path.append(os.path.abspath('../data/nltk_data'))

project_root = os.path.abspath('..')
if project_root not in sys.path:
    sys.path.append(project_root)

from src import text_prep

import pandas as pd
from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import TfidfVectorizer
import joblib

# 1. Ewentualne pobranie zasobów (tylko za pierwszym razem)
# text_prep.download_nltk_resources()

# 2. Pobranie danych 20 Newsgroups
# Warto użyć parametru 'remove', aby usunąć metadane, które psują klastrowanie (model uczyłby się np. adresów email zamiast tematyki)
print("Pobieranie danych...")
dataset = fetch_20newsgroups(subset='all', remove=('headers', 'footers', 'quotes'))
documents = dataset.data

# Zmniejszamy zbiór na czas testów, żeby szybciej się liczyło (opcjonalnie)
documents_subset = documents#[:1000]

# 3. Preprocessing (To potrwa chwilę!)
print("Rozpoczęcie preprocessingu...")
processed_docs = text_prep.preprocess_corpus(documents_subset)
print("Preprocessing zakończony.")

# Podgląd przed i po:
print("\n--- PRZED ---")
print(documents_subset[0][:200])
print("\n--- PO ---")
print(processed_docs[0][:200])

# 4. Reprezentacja TF-IDF
vectorizer = TfidfVectorizer(
    max_df=0.2,        # ignoruj słowa występujące w więcej niż 20% dokumentów
    min_df=5,           # ignoruj słowa występujące w mniej niż 5 dokumentach
    max_features=5000   # ogranicz rozmiar słownika do 5000 najważniejszych słów
)

tfidf_matrix = vectorizer.fit_transform(processed_docs)
print(f"\nKształt macierzy TF-IDF: {tfidf_matrix.shape}")

# 5. Zapisanie przetworzonych danych do późniejszego użytku
os.makedirs('../data', exist_ok=True)
data_to_save = {
    'documents': documents_subset,
    'processed_docs': processed_docs,
    'tfidf_matrix': tfidf_matrix,
    'vectorizer': vectorizer,
    'labels': dataset.target[:1000],
    'target_names': dataset.target_names
}
joblib.dump(data_to_save, '../data/tfidf_data.joblib')
print("\nDane zostały zapisane do pliku: data/tfidf_data.joblib")

Pobieranie danych...
Rozpoczęcie preprocessingu...
Preprocessing zakończony.

--- PRZED ---


I am sure some bashers of Pens fans are pretty confused about the lack
of any kind of posts about the recent Pens massacre of the Devils. Actually,
I am  bit puzzled too and a bit relieved. However,

--- PO ---
sure bashers pen fan pretty confused lack kind post recent pen massacre devil actually bit puzzled bit relieved however going put end non pittsburghers relief bit praise pen man killing devil worse th

Kształt macierzy TF-IDF: (18846, 5000)

Dane zostały zapisane do pliku: data/tfidf_data.joblib
